In [ ]:
import glob, importlib, os, shutil, subprocess, sys
from pathlib import Path
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try: os.unlink(pth)
        except OSError: pass
tmp = Path('/tmp/wheelhouse'); tmp.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower(): continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    t = tmp / name
    if not t.exists(): os.symlink(w, t)
wheels = sorted(str(w) for w in tmp.glob('*.whl'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
print('installed', len(wheels), 'wheels')

In [ ]:
FRESH_SRC = "\"\"\"Per-task pinned environments for the FreshSWE-Py tasks (moshegolderson/freshswe-py) under swegemma's SUBPROCESS sandbox.\n\nWhy: G24 (E4B self-RFT on 37 pool tasks) was null; staff confirmed the hidden test repos are disjoint from the training repos, so more data must\ncome from MORE unseen repos. FreshSWE-Py gives 663 tasks from 33 repos (none of fastapi/rich/requests/httpx), created after Gemma 4's release,\nvalidated under the scorer's conventions with one pinned environment per task (locks/<base_commit>.txt, built for CPython 3.13 in Docker).\n\nOur Kaggle kernels run swegemma with sandbox='subprocess': every sandbox gets a fresh venv that sees the host through `_host_env.pth`, and\n`install_test_dependencies` returns immediately (no sandbox/setup.py runs). So the task environment has to be put on the venv's path by us:\n\n  1. build_store (CPU kernel, internet on, Python 3.12 like the GPU kernels): every distinct `name==version` pin of every lock is installed\n     with `pip install --no-deps --target store/<name>-<version>/site-packages` (FreshSWE's wheelhouse first, then PyPI binary, then sdist);\n     envs.json maps base_commit -> the store dirs of its lock. Packages the host/harness must own (pytest core, pip, setuptools, wheel, the\n     harness's own pytest plugins) are skipped, as swegemma's container setup skips them.\n  2. install_hook (any kernel running the harness): wraps `install_test_dependencies` in every swegemma module that imported it. For a\n     subprocess sandbox it reads the task's base commit from the snapshot's root commit message (`base <sha>`, FreshSWE's convention) and writes\n     `A_freshswe.pth` into the sandbox venv: the workspace and its top-level directories (the competition's sandbox/setup.py fast path), then the\n     task's store dirs. `A_` sorts before `_host_env.pth`, so the task's pins come before any host copy. The hook runs for the agent sandbox and\n     for every verification sandbox (both call install_test_dependencies after extracting the snapshot) and is idempotent.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport concurrent.futures as cf\nimport json\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\n# The host's pytest runs the tests, so pytest itself, its core dependencies, the harness's pytest-timeout and the installers stay the host's.\n# Plugins (pytest-asyncio, pytest-xdist, ...) come from the task's lock: the environment is per task, and the Kaggle host lacks some of them\n# (G25a re-check: strawberry's pytest-aiohttp needs pytest_asyncio).\nSKIP = {\"pytest\", \"pytest-timeout\", \"pluggy\", \"iniconfig\", \"exceptiongroup\", \"tomli\", \"pip\", \"setuptools\", \"wheel\"}\nPTH_NAME = \"A_freshswe.pth\"\n\n\ndef norm(name: str) -> str:\n    return re.sub(r\"[-_.]+\", \"-\", name).lower()\n\n\ndef parse_lock(text: str) -> list[tuple[str, str]]:\n    \"\"\"`name==version` pins from a lock/requirements file; extras, markers, hashes, comments and option lines are dropped.\"\"\"\n    pins = []\n    for raw in text.splitlines():\n        line = raw.split(\"#\", 1)[0].strip()\n        if not line or line.startswith(\"-\") or \"==\" not in line:\n            continue\n        name, ver = line.split(\"==\", 1)\n        name = norm(name.split(\"[\", 1)[0].strip())\n        ver = re.split(r\"[;\\s\\\\]\", ver.strip(), maxsplit=1)[0]\n        if name and ver:\n            pins.append((name, ver))\n    return pins\n\n\ndef store_dir_name(name: str, ver: str) -> str:\n    return f\"{norm(name)}-{ver}\"\n\n\ndef _pip_target(spec: str, target: Path, extra: list[str], timeout: int) -> tuple[int, str]:\n    cmd = [sys.executable, \"-m\", \"pip\", \"install\", \"-q\", \"--disable-pip-version-check\", \"--no-deps\", \"--no-compile\", \"--target\", str(target), *extra, spec]\n    try:\n        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, errors=\"replace\")\n        return p.returncode, (p.stdout + p.stderr)[-600:]\n    except subprocess.TimeoutExpired:\n        return -9, \"timeout\"\n\n\ndef install_pin(name: str, ver: str, store: Path, wheelhouse: Path | None) -> dict:\n    \"\"\"Install one pin into store/<name>-<ver>/site-packages (atomic rename). Returns {'source': wheelhouse|pypi|sdist|cached|None, 'log': ...}.\"\"\"\n    final = store / store_dir_name(name, ver) / \"site-packages\"\n    if final.is_dir():\n        return {\"source\": \"cached\"}\n    spec = f\"{name}=={ver}\"\n    tmp = store / \".tmp\" / store_dir_name(name, ver)\n    attempts = []\n    if wheelhouse is not None:\n        attempts.append((\"wheelhouse\", [\"--only-binary=:all:\", \"--no-index\", \"--find-links\", str(wheelhouse)], 300))\n    attempts += [(\"pypi\", [\"--only-binary=:all:\"], 300), (\"sdist\", [], 600)]\n    log = \"\"\n    for source, extra, timeout in attempts:\n        shutil.rmtree(tmp, ignore_errors=True)\n        tmp.mkdir(parents=True)\n        rc, log = _pip_target(spec, tmp, extra, timeout)\n        if rc == 0:\n            for junk in (\"bin\",):\n                shutil.rmtree(tmp / junk, ignore_errors=True)\n            final.parent.mkdir(parents=True, exist_ok=True)\n            tmp.rename(final)\n            return {\"source\": source}\n    shutil.rmtree(tmp, ignore_errors=True)\n    return {\"source\": None, \"log\": log}\n\n\ndef build_store(tasks: list[dict], locks_dir: Path, store: Path, wheelhouse: Path | None, workers: int = 8, log=print) -> dict:\n    \"\"\"Install every distinct pin of the tasks' locks; return {'envs': {base_commit: [dir names]}, 'missing': {...}, 'stats': {...}}.\"\"\"\n    store.mkdir(parents=True, exist_ok=True)\n    per_task, all_pins, no_lock = {}, set(), []\n    for t in tasks:\n        lock = locks_dir / f\"{t['base_commit']}.txt\"\n        if not lock.exists():\n            no_lock.append(t[\"instance_id\"])\n            continue\n        pins = [(n, v) for n, v in parse_lock(lock.read_text(encoding=\"utf-8\", errors=\"replace\")) if n not in SKIP]\n        per_task[t[\"base_commit\"]] = pins\n        all_pins |= set(pins)\n    log(f\"store: {len(per_task)} locks, {len(all_pins)} distinct pins, {len(no_lock)} tasks without a lock\")\n    results, t0 = {}, time.time()\n    with cf.ThreadPoolExecutor(max_workers=workers) as ex:\n        futs = {ex.submit(install_pin, n, v, store, wheelhouse): (n, v) for n, v in sorted(all_pins)}\n        for i, f in enumerate(cf.as_completed(futs), 1):\n            results[futs[f]] = f.result()\n            if i % 100 == 0 or i == len(futs):\n                log(f\"  pins {i}/{len(futs)} ({time.time() - t0:.0f} s), failed so far {sum(1 for r in results.values() if r['source'] is None)}\")\n    shutil.rmtree(store / \".tmp\", ignore_errors=True)\n    envs, missing = {}, {}\n    for sha, pins in per_task.items():\n        envs[sha] = [store_dir_name(n, v) for n, v in pins if results[(n, v)][\"source\"] is not None]\n        miss = [f\"{n}=={v}\" for n, v in pins if results[(n, v)][\"source\"] is None]\n        if miss:\n            missing[sha] = miss\n    sources = {}\n    for r in results.values():\n        sources[str(r[\"source\"])] = sources.get(str(r[\"source\"]), 0) + 1\n    failed = {f\"{n}=={v}\": r.get(\"log\", \"\")[-300:] for (n, v), r in results.items() if r[\"source\"] is None}\n    return {\"envs\": envs, \"missing\": missing, \"failed_pins\": failed, \"no_lock\": no_lock,\n            \"stats\": {\"locks\": len(per_task), \"pins\": len(all_pins), \"sources\": sources, \"seconds\": round(time.time() - t0)}}\n\n\ndef base_commit_of(workspace: Path) -> str | None:\n    \"\"\"FreshSWE snapshots hold one root commit whose subject is `base <sha>`; later commits (the harness baseline) sit on top of it.\"\"\"\n    try:\n        out = subprocess.run([\"git\", \"-C\", str(workspace), \"log\", \"--format=%s\"], capture_output=True, text=True, timeout=60).stdout\n    except (OSError, subprocess.TimeoutExpired):\n        return None\n    for line in out.splitlines()[::-1]:\n        parts = line.split()\n        if len(parts) == 2 and parts[0] == \"base\" and len(parts[1]) == 40:\n            return parts[1]\n    return None\n\n\ndef pth_lines(workspace: Path, store: Path, dirs: list[str]) -> list[str]:\n    ws = [str(workspace)] + [str(p) for p in sorted(workspace.iterdir()) if p.is_dir() and not p.name.startswith(\".\")]\n    return ws + [str(store / d / \"site-packages\") for d in dirs]\n\n\nPRETEND_VERSION = \"99.0.0\"   # FreshSWE's image sets SETUPTOOLS_SCM_PRETEND_VERSION=99.0.0\nSCM_TEMPLATE = (\"# generated by fresh_env (stands in for the editable install's setuptools-scm / hatch-vcs version file)\\n\"\n                \"__version__ = version = '{v}'\\n__version_tuple__ = version_tuple = ({t})\\n__commit_id__ = commit_id = None\\n\")\n\n\ndef project_meta(ws: Path) -> dict:\n    \"\"\"What an editable install would provide: the distribution name and version, and the version file a VCS build hook writes.\"\"\"\n    import configparser  # noqa: PLC0415\n    import tomllib  # noqa: PLC0415\n    meta = {\"name\": None, \"version\": None, \"version_files\": []}\n    pp = ws / \"pyproject.toml\"\n    if pp.is_file():\n        try:\n            d = tomllib.loads(pp.read_text(encoding=\"utf-8\", errors=\"replace\"))\n        except Exception:  # noqa: BLE001\n            d = {}\n        proj, tool = d.get(\"project\", {}) or {}, d.get(\"tool\", {}) or {}\n        meta[\"name\"] = proj.get(\"name\") or (tool.get(\"poetry\", {}) or {}).get(\"name\")\n        v = proj.get(\"version\") or (tool.get(\"poetry\", {}) or {}).get(\"version\")\n        meta[\"version\"] = v if isinstance(v, str) and \"version\" not in (proj.get(\"dynamic\") or []) else None\n        vcs = (((tool.get(\"hatch\", {}) or {}).get(\"build\", {}) or {}).get(\"hooks\", {}) or {}).get(\"vcs\", {}) or {}\n        scm = tool.get(\"setuptools_scm\", tool.get(\"setuptools-scm\", {})) or {}\n        pdm = ((tool.get(\"pdm\", {}) or {}).get(\"version\", {}) or {})\n        for f in (vcs.get(\"version-file\"), vcs.get(\"version_file\"), scm.get(\"version_file\"), scm.get(\"write_to\"), pdm.get(\"write_to\")):\n            if isinstance(f, str) and f.endswith(\".py\"):\n                meta[\"version_files\"].append(f)\n    sc = ws / \"setup.cfg\"\n    if not meta[\"name\"] and sc.is_file():\n        cp = configparser.ConfigParser()\n        try:\n            cp.read_string(sc.read_text(encoding=\"utf-8\", errors=\"replace\"))\n            meta[\"name\"] = cp.get(\"metadata\", \"name\", fallback=None)\n        except configparser.Error:\n            pass\n    return meta\n\n\ndef materialize_project(ws: Path, site_packages: list[Path]) -> str:\n    \"\"\"Emulate the parts of `pip install -e .` the tests rely on: the VCS version file (only if absent) and a minimal .dist-info\n    (so importlib.metadata.version(<project>) works) in the sandbox venv. Returns a short status.\"\"\"\n    m = project_meta(ws)\n    ver = m[\"version\"] or PRETEND_VERSION\n    wrote = []\n    for rel in m[\"version_files\"]:\n        for cand in (ws / rel, ws / \"src\" / rel):\n            if cand.parent.is_dir():\n                if not cand.exists():\n                    cand.write_text(SCM_TEMPLATE.format(v=PRETEND_VERSION, t=\", \".join(PRETEND_VERSION.split(\".\"))), encoding=\"utf-8\")\n                    wrote.append(rel)\n                break\n    if m[\"name\"]:\n        dist = re.sub(r\"[-_.]+\", \"_\", m[\"name\"])\n        for sp in site_packages:\n            di = sp / f\"{dist}-{ver}.dist-info\"\n            di.mkdir(exist_ok=True)\n            (di / \"METADATA\").write_text(f\"Metadata-Version: 2.1\\nName: {m['name']}\\nVersion: {ver}\\n\", encoding=\"utf-8\")\n            (di / \"INSTALLER\").write_text(\"fresh_env\\n\", encoding=\"utf-8\")\n    return f\"name={m['name']},ver={ver},vfiles={len(wrote)}\"\n\n\ndef write_env_pth(paths: dict, store: Path, envs: dict) -> str:\n    \"\"\"Write A_freshswe.pth into the sandbox venv. Returns a short status string (for logging/tests).\"\"\"\n    workspace, venv = Path(paths[\"workspace\"]), Path(paths[\"venv\"])\n    sha = base_commit_of(workspace)\n    if sha is None:\n        return \"no-base-commit\"\n    if sha not in envs:\n        return f\"no-env:{sha[:10]}\"\n    sps = list(venv.glob(\"lib/python*/site-packages\"))\n    materialize_project(workspace, sps)   # before pth_lines: may not create top-level dirs, only files inside existing ones\n    lines = pth_lines(workspace, store, envs[sha])\n    for sp in sps:\n        (sp / PTH_NAME).write_text(\"\\n\".join(lines) + \"\\n\", encoding=\"utf-8\")\n    return f\"ok:{sha[:10]}:{len(envs[sha])}:{len(sps)}\"\n\n\ndef install_hook(store: Path, envs: dict, log=None) -> list[str]:\n    \"\"\"Wrap install_test_dependencies in every loaded swegemma module. Returns the patched module names.\"\"\"\n    import swegemma.harness.container_setup as cs  # noqa: PLC0415\n\n    orig = getattr(cs.install_test_dependencies, \"_fresh_orig\", cs.install_test_dependencies)\n    stats = install_hook.stats = {}\n\n    def hooked(docker, container_id, *args, **kwargs):\n        try:\n            if type(docker).__name__ in {\"SubprocessManager\", \"SubprocessSandbox\"} and hasattr(docker, \"sandboxes\"):\n                status = write_env_pth(docker.sandboxes[container_id], store, envs)\n                key = status.split(\":\", 1)[0]\n                stats[key] = stats.get(key, 0) + 1\n                if log and key != \"ok\":\n                    log(f\"fresh_env hook: {status}\")\n        except Exception as e:  # never break the harness; the task then runs on the host environment and is reported by the gold check\n            stats[\"error\"] = stats.get(\"error\", 0) + 1\n            if log:\n                log(f\"fresh_env hook error: {type(e).__name__}: {e}\")\n        return orig(docker, container_id, *args, **kwargs)\n\n    hooked._fresh_orig = orig\n    patched = []\n    for name, mod in list(sys.modules.items()):\n        fn = getattr(mod, \"install_test_dependencies\", None) if name.startswith(\"swegemma\") and mod is not None else None\n        if fn is not None and (fn is orig or getattr(fn, \"_fresh_orig\", None) is orig):\n            mod.install_test_dependencies = hooked\n            patched.append(name)\n    return patched\n\n\nif __name__ == \"__main__\":\n    print(json.dumps(parse_lock(Path(sys.argv[1]).read_text()), indent=0)[:2000])\n"

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
WORK = Path('/kaggle/working'); Path('/kaggle/working/fresh_env.py').write_text(FRESH_SRC, encoding='utf-8'); sys.path.insert(0, '/kaggle/working')
import fresh_env as fe
FRESH = next(p.parent for p in Path('/kaggle/input').rglob('tasks.jsonl') if (p.parent / 'locks').is_dir())
WH = FRESH / 'wheelhouse' if (FRESH / 'wheelhouse').is_dir() else None
TASKS = [json.loads(l) for l in open(FRESH / 'tasks.jsonl', encoding='utf-8')]
LIMIT, IDS = 0, []
if IDS:
    TASKS = [t for t in TASKS if t['instance_id'] in set(IDS)]
elif LIMIT:   # spread a small run over repos
    by = {}
    for t in TASKS: by.setdefault(t['repo'], []).append(t)
    TASKS = [t for i in range(LIMIT) for ts in by.values() if i < len(ts) for t in [ts[i]]][:LIMIT]
print('python', sys.version.split()[0], '| FreshSWE at', FRESH, '| wheelhouse', WH, len(list(WH.glob('*.whl'))) if WH else 0, 'wheels', '| tasks', len(TASKS), '| snapshots', len(list((FRESH / 'snapshots').glob('*.tgz'))))
print('top-level:', sorted(p.name for p in FRESH.iterdir()))
STORE = WORK / 'fresh_store'
T0 = time.time()
rep = fe.build_store(TASKS, FRESH / 'locks', STORE, WH, workers=8)
(WORK / 'envs.json').write_text(json.dumps(rep['envs']), encoding='utf-8')
(WORK / 'build_report.json').write_text(json.dumps({k: v for k, v in rep.items() if k != 'envs'}, indent=1), encoding='utf-8')
du = subprocess.run(['du', '-sm', str(STORE)], capture_output=True, text=True).stdout.split()[0]
print('STORE', rep['stats'], '| MB', du, '| tasks with missing pins', len(rep['missing']), '| no lock', len(rep['no_lock']))
print('failed pins (first 30):', list(rep['failed_pins'])[:30])

In [ ]:
import asyncio, logging, traceback
from adk_eval_core.tracing import SessionTrace
from adk_submission import ModelRegistry, discover_declared_models
from google.adk.models.lite_llm import LiteLlm
from swegemma.config import EvalConfig
from swegemma.deduplication import resolve_task_snapshot_paths
from swegemma.evaluate import Evaluator
import swegemma.harness.verification
from swegemma.harness.verification import verify_task
from swegemma.models import load_tasks

class _DropEnsurepipNoise(logging.Filter):
    def filter(self, record): return 'ensurepip unavailable' not in record.getMessage()
logging.getLogger('swegemma.sandbox.subprocess').addFilter(_DropEnsurepipNoise())

patched = fe.install_hook(STORE, rep['envs'], log=print)
print('hook installed in', patched)
assert 'swegemma.harness.verification' in patched, patched
ARM = next(p.parent for p in Path('/kaggle/input').rglob('agent.yaml') if 'swe_scaffold_v2_b1' in p.read_text(encoding='utf-8'))
(WORK / 'no_wheels').mkdir(exist_ok=True); (WORK / 'gold').mkdir(exist_ok=True)
SEL = WORK / 'tasks_selected.jsonl'
with open(SEL, 'w', encoding='utf-8') as f:
    for t in TASKS: f.write(json.dumps(t) + '\n')
reg = ModelRegistry()
for m in discover_declared_models(ARM): reg.register(m, LiteLlm(model='openai/' + m, api_base='http://127.0.0.1:8000/v1', api_key='EMPTY'))
config = EvalConfig(tasks_path=SEL, snapshots_dir=FRESH / 'snapshots', results_dir=WORK / 'gold', submission_dir=ARM, models=reg, sandbox='subprocess',
    graph_dir=str(WORK / 'no_graphs'), embeddings_dir=str(WORK / 'no_embeddings'), wheels_dir=WORK / 'no_wheels')
evaluator = Evaluator(config)
tasks = load_tasks(config.tasks_path)
print('tasks loaded by swegemma:', len(tasks), flush=True)
results = {}
sem = asyncio.Semaphore(4)
BUDGET_S = 7.0 * 3600; TG = time.time()

async def one(task, patch):
    t0 = time.perf_counter()
    try:
        snap, base, patch_path = resolve_task_snapshot_paths(config.snapshots_dir, task.instance_id, task.repo)
        res = await verify_task(evaluator.docker, config, task, snap, base_snapshot_path=base, patch_path=patch_path, agent_patch=patch, agent_error=None,
                                trace=SessionTrace(), start_time=t0)
        return {'resolved': bool(res.resolved), 'error': res.error, 'exit': res.test_exit_code, 'tail': (res.test_output or '')[-1500:] if not res.resolved else '', 'sec': round(time.perf_counter() - t0, 1)}
    except Exception as e:
        return {'resolved': False, 'error': f'{type(e).__name__}: {e}'[:400], 'exit': -1, 'tail': traceback.format_exc()[-800:], 'sec': round(time.perf_counter() - t0, 1)}

async def check(task):
    async with sem:
        if time.time() - TG > BUDGET_S:
            results[task.instance_id] = {'repo': task.repo, 'gold': {'resolved': False, 'error': 'skipped_budget'}}; return
        g = await one(task, task.patch)
        rec = {'repo': task.repo, 'gold': g}
        if g['resolved']:
            rec['empty'] = await one(task, '')
        results[task.instance_id] = rec
        print(f"[{len(results)}/{len(tasks)}] {task.instance_id}: gold {'PASS' if g['resolved'] else 'FAIL'} ({g['sec']}s)" + (f" | empty {'RESOLVED(!)' if rec['empty']['resolved'] else 'fails (good)'}" if 'empty' in rec else '') + f" | hook {fe.install_hook.stats}", flush=True)
        if len(results) % 20 == 0: (WORK / 'gold_results.json').write_text(json.dumps(results, indent=1), encoding='utf-8')

async def main():
    await asyncio.gather(*(check(t) for t in tasks))

try:
    loop = asyncio.get_running_loop()
except RuntimeError:
    loop = None
if loop is not None and loop.is_running():
    import concurrent.futures
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(lambda: asyncio.run(main())).result()
else:
    asyncio.run(main())

(WORK / 'gold_results.json').write_text(json.dumps(results, indent=1), encoding='utf-8')
good = sorted(i for i, r in results.items() if r['gold']['resolved'] and 'empty' in r and not r['empty']['resolved'])
(WORK / 'rollout_pool.json').write_text(json.dumps(good), encoding='utf-8')
print('\n===== G25a: harness gold check on FreshSWE-Py (subprocess sandbox + per-task pinned env) =====')
print(f"gold passes {sum(r['gold']['resolved'] for r in results.values())}/{len(results)} | empty patch also resolved {sum(1 for r in results.values() if 'empty' in r and r['empty']['resolved'])} | USABLE {len(good)} tasks, {len({results[i]['repo'] for i in good})} repos | hook {fe.install_hook.stats} | gold stage {(time.time() - TG) / 60:.0f} min")
per = {}
for i, r in results.items():
    a = per.setdefault(r['repo'], [0, 0]); a[0] += 1; a[1] += i in good
print('per repo usable/checked:', {k: f'{v[1]}/{v[0]}' for k, v in sorted(per.items(), key=lambda kv: -kv[1][0])})
errs = {}
for i, r in results.items():
    if not r['gold']['resolved']: errs.setdefault((r['gold']['error'] or 'tests failed')[:90], []).append(i)
print('\nFAIL reasons:')
for k, v in sorted(errs.items(), key=lambda kv: -len(kv[1]))[:20]: print(f'  {len(v):3d}  {k}   e.g. {v[:3]}')
fails = [i for i, r in results.items() if not r['gold']['resolved'] and r['gold']['error'] is None][:6]
for i in fails: print('\n---', i, '\n', results[i]['gold']['tail'][-900:])
# package the store as one file (thousands of small files are slow as kernel output), then drop the directory
subprocess.run(f'tar -C {WORK} -cf - fresh_store | pigz -p 4 > {WORK}/fresh_store.tar.gz || tar -C {WORK} -czf {WORK}/fresh_store.tar.gz fresh_store', shell=True, check=True)
print('tar MB', round((WORK / 'fresh_store.tar.gz').stat().st_size / 1e6))
import shutil; shutil.rmtree(STORE); shutil.rmtree(WORK / 'gold', ignore_errors=True)
print('total minutes', round((time.time() - T0) / 60))